# AI TraceFinder Master Colab Notebook
## Forensic Scanner Identification Pipeline

This notebook provides the complete end-to-end workflow for **AI TraceFinder**, including dataset preprocessing, metadata feature extraction, baseline model training (Random Forest & SVM), evaluation, and model artifact saving.

In [ ]:
# 1. Environment Setup & Imports
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Ensure project root is in python path
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(PROJECT_ROOT))

print(f"Project root initialized at: {PROJECT_ROOT}")

### 2. Load Processed Metadata Features

In [ ]:
csv_path = PROJECT_ROOT / 'data' / 'metadata_features.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"Loaded metadata dataset with shape: {df.shape}")
    display(df.head())
else:
    print(f"Dataset not found at {csv_path}. Please execute preprocessing scripts first.")

### 3. Model Training & Evaluation

In [ ]:
if 'df' in locals():
    feature_cols = [c for c in df.columns if c not in ['file_name', 'main_class', 'resolution', 'class_label']]
    X = df[feature_cols]
    y = df['class_label']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    rf = RandomForestClassifier(n_estimators=300, random_state=42)
    rf.fit(X_train_scaled, y_train)
    y_pred_rf = rf.predict(X_test_scaled)

    print("=== Random Forest Results ===")
    print(classification_report(y_test, y_pred_rf))

    # Save models
    models_dir = PROJECT_ROOT / 'models'
    models_dir.mkdir(exist_ok=True)
    joblib.dump(rf, models_dir / 'random_forest.pkl')
    joblib.dump(scaler, models_dir / 'scaler.pkl')
    print("Model artifacts saved successfully.")